# NAV Cleaning
Clean `02_nav_history.csv` by converting `date` to datetime, sorting by `amfi_code` and `date`, removing duplicates, checking `nav > 0`, and saving to `data/processed/clean_nav.csv`.


In [ ]:
from pathlib import Path
import csv
from datetime import datetime

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent

raw_path = repo_root / 'data' / 'raw' / '02_nav_history.csv'
out_path = repo_root / 'data' / 'processed' / 'clean_nav.csv'
out_path.parent.mkdir(parents=True, exist_ok=True)

rows = []
with raw_path.open(newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        try:
            row_date = datetime.fromisoformat(row['date'])
            nav = float(row['nav'])
        except Exception:
            continue
        if nav <= 0:
            continue
        rows.append((row['amfi_code'], row_date, nav))

seen = set()
unique_rows = []
for amfi_code, row_date, nav in rows:
    key = (amfi_code, row_date, nav)
    if key in seen:
        continue
    seen.add(key)
    unique_rows.append((amfi_code, row_date, nav))

unique_rows.sort(key=lambda x: (x[0], x[1]))

with out_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['amfi_code', 'date', 'nav'])
    for amfi_code, row_date, nav in unique_rows:
        writer.writerow([amfi_code, row_date.date().isoformat(), f'{nav:.6f}'.rstrip('0').rstrip('.')])

print(f'Saved {len(unique_rows)} cleaned rows to {out_path}')


# Investor Transaction Cleaning
Clean `08_investor_transactions.csv` by standardizing transaction types, verifying `amount_inr > 0`, fixing date formats, and saving to `data/processed/clean_transactions.csv`.


In [ ]:
from pathlib import Path
import csv
from datetime import datetime

def normalize_transaction_type(value):
    if value is None:
        return ''
    text = value.strip().upper()
    mapping = {
        'SIP': 'SIP',
        'SWP': 'SWP',
        'STP': 'STP',
        'RED': 'REDEMPTION',
        'REDEMPTION': 'REDEMPTION',
        'SELL': 'REDEMPTION',
        'PURCHASE': 'PURCHASE',
        'BUY': 'PURCHASE',
        'LUMPSUM': 'LUMPSUM',
        'SWITCH': 'SWITCH',
        'TRANSFER': 'TRANSFER',
    }
    return mapping.get(text, text)

def parse_date(value):
    if value is None:
        return None
    text = value.strip()
    if not text:
        return None
    formats = ['%Y-%m-%d', '%d-%m-%Y', '%d/%m/%Y', '%d-%b-%Y', '%d %b %Y']
    for fmt in formats:
        try:
            return datetime.strptime(text, fmt).date()
        except Exception:
            continue
    try:
        return datetime.fromisoformat(text).date()
    except Exception:
        return None

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent

raw_path = repo_root / 'data' / 'raw' / '08_investor_transactions.csv'
out_path = repo_root / 'data' / 'processed' / 'clean_transactions.csv'
out_path.parent.mkdir(parents=True, exist_ok=True)

cleaned = []
with raw_path.open(newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        transaction_date = parse_date(row.get('transaction_date'))
        if transaction_date is None:
            continue
        try:
            amount = float(row.get('amount_inr', '').replace(',', ''))
        except Exception:
            continue
        if amount <= 0:
            continue
        transaction_type = normalize_transaction_type(row.get('transaction_type'))
        cleaned.append({
            'investor_id': row.get('investor_id', '').strip(),
            'transaction_date': transaction_date.isoformat(),
            'amfi_code': row.get('amfi_code', '').strip(),
            'transaction_type': transaction_type,
            'amount_inr': f'{amount:.2f}',
            'state': row.get('state', '').strip(),
            'city': row.get('city', '').strip(),
            'city_tier': row.get('city_tier', '').strip(),
            'age_group': row.get('age_group', '').strip(),
            'gender': row.get('gender', '').strip(),
            'annual_income_lakh': row.get('annual_income_lakh', '').strip(),
            'payment_mode': row.get('payment_mode', '').strip(),
            'kyc_status': row.get('kyc_status', '').strip(),
        })

fieldnames = ['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']
with out_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(cleaned)

print(f'Saved {len(cleaned)} cleaned transaction rows to {out_path}')


# Scheme Performance Cleaning
Clean `07_scheme_performance.csv` by validating numeric columns, checking Sharpe ratios, and saving to `data/processed/clean_performance.csv`.


In [ ]:
from pathlib import Path
import csv

numeric_fields = [
    'return_1yr_pct',
    'return_3yr_pct',
    'return_5yr_pct',
    'benchmark_3yr_pct',
    'alpha',
    'beta',
    'sharpe_ratio',
    'sortino_ratio',
    'std_dev_ann_pct',
    'max_drawdown_pct',
    'aum_crore',
    'expense_ratio_pct',
]

def parse_float(value):
    if value is None:
        return None
    text = str(value).strip()
    if not text:
        return None
    try:
        return float(text.replace(',', ''))
    except Exception:
        return None

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent

raw_path = repo_root / 'data' / 'raw' / '07_scheme_performance.csv'
out_path = repo_root / 'data' / 'processed' / 'clean_performance.csv'
out_path.parent.mkdir(parents=True, exist_ok=True)

cleaned = []
with raw_path.open(newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        parsed = {}
        valid = True
        for field in numeric_fields:
            parsed_value = parse_float(row.get(field))
            if parsed_value is None:
                valid = False
                break
            parsed[field] = parsed_value
        if not valid:
            continue
        if parsed['sharpe_ratio'] is None:
            continue
        entry = {
            'amfi_code': row.get('amfi_code', '').strip(),
            'scheme_name': row.get('scheme_name', '').strip(),
            'fund_house': row.get('fund_house', '').strip(),
            'category': row.get('category', '').strip(),
            'plan': row.get('plan', '').strip(),
        }
        entry.update({field: parsed[field] for field in numeric_fields})
        entry['morningstar_rating'] = row.get('morningstar_rating', '').strip()
        entry['risk_grade'] = row.get('risk_grade', '').strip()
        cleaned.append(entry)

fieldnames = ['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan'] + numeric_fields + ['morningstar_rating', 'risk_grade']
with out_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(cleaned)

print(f'Saved {len(cleaned)} cleaned scheme performance rows to {out_path}')
